In [1]:
import os

In [2]:
%pwd

'c:\\Users\\vishn\\Desktop\\NLP\\End-to-end-Text-Summarizer\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'c:\\Users\\vishn\\Desktop\\NLP\\End-to-end-Text-Summarizer'

In [5]:
import os
from pathlib import Path

os.chdir(r"C:\Users\vishn\Desktop\NLP\End-to-end-Text-Summarizer")

print(os.getcwd())
print(Path("config/config.yaml").exists())
print(Path("params.yaml").exists())

C:\Users\vishn\Desktop\NLP\End-to-end-Text-Summarizer
True
True


In [6]:
from pathlib import Path

ROOT_DIR = Path(r"C:\Users\vishn\Desktop\NLP\End-to-end-Text-Summarizer")

CONFIG_FILE_PATH = ROOT_DIR / "config" / "config.yaml"
PARAMS_FILE_PATH = ROOT_DIR / "params.yaml"

In [7]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int

In [8]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [9]:
def get_model_trainer_config(self) -> ModelTrainerConfig:

    config = self.config.model_trainer
    params = self.params.TrainingArguments

    create_directories([config.root_dir])

    model_trainer_config = ModelTrainerConfig(
        root_dir=config.root_dir,
        data_path=config.data_path,
        model_ckpt=config.model_ckpt,

        num_train_epochs=params.num_train_epochs,
        warmup_steps=params.warmup_steps,

        per_device_train_batch_size=params.per_device_train_batch_size,

        weight_decay=params.weight_decay,
        logging_steps=params.logging_steps,

        evaluation_strategy=params.evaluation_strategy,
        eval_steps=params.eval_steps,

        save_steps=params.save_steps,
        gradient_accumulation_steps=params.gradient_accumulation_steps
    )

    return model_trainer_config

In [10]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch

c:\Users\vishn\Desktop\NLP\End-to-end-Text-Summarizer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig) -> None:
        self.config = config

    def train(self):
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model = model_pegasus)

        data_samsum_pt = load_from_disk(self.config.data_path)

        from transformers import TrainingArguments, Trainer

        training_args = TrainingArguments(
            output_dir=self.config.root_dir,
            num_train_epochs=self.config.num_train_epochs,
            warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            per_device_eval_batch_size=self.config.per_device_eval_batch_size,
            weight_decay=self.config.weight_decay,
            logging_steps=self.config.logging_steps,
            eval_strategy=self.config.eval_strategy,
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps
        )

        trainer.train()

        model_pegasus.save_pretrained(os.path.join(self.config.root_dir,'pegasus-samsum-model'))

        tokenizer.save_pretrained(os.path.join(self.config.root_dir,'tokenizer'))

In [14]:
print(ConfigurationManager)

<class 'textSummarizer.config.configuration.ConfigurationManager'>


In [17]:
from textSummarizer.config.configuration import ConfigurationManager

print(dir(ConfigurationManager))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', 'get_data_ingestion_config', 'get_data_transformation_config', 'get_data_validation_config']


In [ ]:
try:
    config = ConfigurationManager()

    trainer_config = config.get_model_trainer_config()

    trainer = ModelTrainer(config=trainer_config)

    trainer.train()

except Exception as e:
    raise e

[2026-06-08 09:17:00,316: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-08 09:17:00,319: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-08 09:17:00,321: INFO: common: created directory at: artifacts]


AttributeError: 'ConfigurationManager' object has no attribute 'get_model_trainer_config'